In [0]:
--------Ejercicio 1: El "Ritmo de Carrera" (Promedio de vueltas limpias)-------------
SELECT
    get_json_object(dri.raw_json, '$.full_name') AS full_name,
    get_json_object(dri.raw_json, '$.team_name') AS team_name,
    ROUND(AVG(CAST(get_json_object(lap.raw_json, '$.lap_duration') AS DOUBLE)), 2) AS avg_lap_duration
FROM
    workspace.formula_1.bronze_laps lap
JOIN 
    workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
JOIN 
    workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
JOIN
    workspace.formula_1.bronze_drivers dri ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
        AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
        AND get_json_object(ses.raw_json, '$.session_key') = get_json_object(dri.raw_json, '$.session_key')
WHERE
    get_json_object(ses.raw_json, '$.session_type') = 'Race'
    AND
    get_json_object(lap.raw_json, '$.is_pit_out_lap') = 'false'
    AND 
    get_json_object(met.raw_json, '$.country_name') = 'Monaco'
GROUP BY
    get_json_object(dri.raw_json, '$.full_name'),
    get_json_object(dri.raw_json, '$.team_name')
ORDER BY
    avg_lap_duration ASC
;


--------Ejercicio 2: Consistencia entre Sectores--------
SELECT
    get_json_object(dri.raw_json, '$.full_name') AS full_name,
    get_json_object(dri.raw_json, '$.team_name') AS team_name,
    COUNT(CAST(get_json_object(lap.raw_json, '$.lap_number') AS INT)) AS total_laps,
    ROUND(STDDEV(CAST(get_json_object(lap.raw_json, '$.duration_sector_1') AS DOUBLE)), 3) AS std_dev_sector_1,
    ROUND(STDDEV(CAST(get_json_object(lap.raw_json, '$.duration_sector_2') AS DOUBLE)), 3) AS std_dev_sector_2
FROM
    workspace.formula_1.bronze_laps lap
JOIN 
    workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
JOIN 
    workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
JOIN
    workspace.formula_1.bronze_drivers dri ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
        AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
        AND get_json_object(ses.raw_json, '$.session_key') = get_json_object(dri.raw_json, '$.session_key')
WHERE
    get_json_object(lap.raw_json, '$.is_pit_out_lap') = 'false'
    AND 
    get_json_object(ses.raw_json, '$.session_key') = '9686'
GROUP BY
    get_json_object(dri.raw_json, '$.full_name'),
    get_json_object(dri.raw_json, '$.team_name')
HAVING
    total_laps > 10
ORDER BY
    std_dev_sector_1, std_dev_sector_2 ASC
;


---------------Ejercicio 3: Evolución de las posiciones en "Tiempo Real"-----------------
WITH cumulative_lap_duration AS (
    SELECT
        CAST(get_json_object(lap.raw_json, '$.lap_number') AS INT) AS lap_number,
        get_json_object(dri.raw_json, '$.full_name') AS full_name,
        get_json_object(dri.raw_json, '$.team_name') AS team_name,
        SUM(CAST(get_json_object(lap.raw_json, '$.lap_duration') AS DOUBLE)) OVER (
            PARTITION BY get_json_object(ses.raw_json, '$.session_key'), get_json_object(dri.raw_json, '$.driver_number')
            ORDER BY CAST(get_json_object(lap.raw_json, '$.lap_number') AS INT)
        ) AS race_accumulated_time
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
    JOIN 
        workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
    JOIN
        workspace.formula_1.bronze_drivers dri ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
            AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
            AND get_json_object(ses.raw_json, '$.session_key') = get_json_object(dri.raw_json, '$.session_key')
    WHERE
        get_json_object(lap.raw_json, '$.lap_duration') IS NOT NULL
        AND 
        get_json_object(ses.raw_json, '$.session_key') = '9686'
),
driver_ranking AS (
    SELECT
        *,
        DENSE_RANK() OVER(PARTITION BY lap_number ORDER BY race_accumulated_time ASC) AS driver_rank
    FROM
        cumulative_lap_duration
)
SELECT
    *
FROM
    driver_ranking
ORDER BY 
    lap_number ASC, driver_rank ASC
;

---------Ejercicio 4: Análisis de la "Vuelta Rápida" del Gran Premio-------------
WITH drivers_ranking AS (
    SELECT
        get_json_object(met.raw_json, '$.circuit_short_name') AS circuit_short_name,
        get_json_object(dri.raw_json, '$.full_name') AS full_name,
        get_json_object(dri.raw_json, '$.team_name') AS team_name,
        CAST(get_json_object(lap.raw_json, '$.lap_number') AS INT) AS lap_number,
        CAST(get_json_object(lap.raw_json, '$.lap_duration') AS DOUBLE) AS lap_duration,
        ROW_NUMBER() OVER(PARTITION BY get_json_object(met.raw_json, '$.circuit_short_name') ORDER BY CAST(get_json_object(lap.raw_json, '$.lap_duration') AS DOUBLE) ASC) AS row_num
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
    JOIN 
        workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
    JOIN
        workspace.formula_1.bronze_drivers dri ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
            AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
            AND get_json_object(ses.raw_json, '$.session_key') = get_json_object(dri.raw_json, '$.session_key')
    WHERE
        get_json_object(lap.raw_json, '$.is_pit_out_lap') = 'false'
        AND 
        get_json_object(lap.raw_json, '$.lap_duration') IS NOT NULL
)
SELECT
    circuit_short_name,
    full_name,
    team_name,
    lap_number,
    lap_duration
FROM
    drivers_ranking
WHERE
    row_num = 1
ORDER BY
    circuit_short_name,
    lap_duration ASC
;

------------Ejercicio 5: Detección de "Anomalías" en Carrera (Safety Car / Incidentes)------------
WITH lap_driver_anomaly AS 
(
    SELECT
        get_json_object(dri.raw_json, '$.full_name') AS full_name,
        CAST(get_json_object(lap.raw_json, '$.lap_number') AS INT) AS lap_number,
        CAST(get_json_object(lap.raw_json, '$.duration_sector_3') AS DOUBLE) AS duration_sector_3,
        AVG(CAST(get_json_object(lap.raw_json, '$.duration_sector_3') AS DOUBLE)) OVER(
            PARTITION BY get_json_object(ses.raw_json, '$.session_key'), get_json_object(dri.raw_json, '$.driver_number')
        ) AS promedio
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
    JOIN 
        workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
    JOIN
        workspace.formula_1.bronze_drivers dri ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
            AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
            AND get_json_object(ses.raw_json, '$.session_key') = get_json_object(dri.raw_json, '$.session_key')
    WHERE
        get_json_object(lap.raw_json, '$.is_pit_out_lap') = 'false'
        AND 
        get_json_object(lap.raw_json, '$.duration_sector_3') IS NOT NULL
        AND
        get_json_object(ses.raw_json, '$.session_key') = '9686'
)
SELECT
    full_name,
    lap_number,
    duration_sector_3,
    ROUND(promedio, 3) AS driver_avg_sector_3
FROM
    lap_driver_anomaly
WHERE
    duration_sector_3 >= promedio * 1.5
ORDER BY
    full_name ASC,
    lap_number ASC
;

------------Ejercicio 6: El "Piloto Ideal" (Vuelta Teórica Perfecta)------------
WITH ideal_pilot AS (
    SELECT
        get_json_object(dri.raw_json, '$.full_name') AS full_name,
        get_json_object(dri.raw_json, '$.team_name') AS team_name,
        MIN(CAST(get_json_object(lap.raw_json, '$.duration_sector_1') AS DOUBLE)) + MIN(CAST(get_json_object(lap.raw_json, '$.duration_sector_2') AS DOUBLE)) + MIN(CAST(get_json_object(lap.raw_json, '$.duration_sector_3') AS DOUBLE)) AS theoretical_best_lap,
        MIN(CAST(get_json_object(lap.raw_json, '$.lap_duration') AS DOUBLE)) AS actual_best_lap
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
    JOIN 
        workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
    JOIN
        workspace.formula_1.bronze_drivers dri ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
            AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
            AND get_json_object(ses.raw_json, '$.session_key') = get_json_object(dri.raw_json, '$.session_key')
    WHERE
        get_json_object(lap.raw_json, '$.is_pit_out_lap') = 'false'
        AND 
        get_json_object(lap.raw_json, '$.duration_sector_1') IS NOT NULL
        AND 
        get_json_object(lap.raw_json, '$.duration_sector_2') IS NOT NULL
        AND 
        get_json_object(lap.raw_json, '$.duration_sector_3') IS NOT NULL
        AND 
        get_json_object(lap.raw_json, '$.lap_duration') IS NOT NULL
        AND
        get_json_object(ses.raw_json, '$.session_key') = '9686'
    GROUP BY
        get_json_object(dri.raw_json, '$.full_name'),
        get_json_object(dri.raw_json, '$.team_name')
)
SELECT
    full_name,
    team_name,
    ROUND(theoretical_best_lap, 3) AS theoretical_best_lap,
    actual_best_lap,
    ROUND(actual_best_lap - theoretical_best_lap, 3) AS time_gap
FROM
    ideal_pilot
ORDER BY
    time_gap ASC;


------------Ejercicio 7: Ranking de pilotos por meeting------------
with meeting_driver_ranking AS (
    SELECT
        get_json_object(dri.raw_json, '$.full_name') AS full_name,
        get_json_object(dri.raw_json, '$.team_name') AS team_name,
        get_json_object(met.raw_json, '$.meeting_key') AS meeting_key,
        get_json_object(met.raw_json, '$.meeting_official_name') AS meeting_name,
        ROUND(
            SUM(TRY_CAST(get_json_object(lap.raw_json, '$.lap_duration') AS DOUBLE)), 
        2) AS total_lap_duration,
        COUNT(get_json_object(lap.raw_json, '$.lap_number')) AS total_laps
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
    JOIN 
        workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
    JOIN
        workspace.formula_1.bronze_drivers dri ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
            AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
            AND get_json_object(ses.raw_json, '$.session_key') = get_json_object(dri.raw_json, '$.session_key')
    WHERE
        get_json_object(lap.raw_json, '$.is_pit_out_lap') = 'false'
        AND 
        get_json_object(lap.raw_json, '$.duration_sector_1') IS NOT NULL
        AND 
        get_json_object(lap.raw_json, '$.duration_sector_2') IS NOT NULL
        AND 
        get_json_object(lap.raw_json, '$.duration_sector_3') IS NOT NULL
        AND 
        get_json_object(lap.raw_json, '$.lap_duration') IS NOT NULL
    GROUP BY
        full_name,
        team_name,
        meeting_key,
        meeting_name
    HAVING
        total_laps > 50
)
SELECT
    meeting_name,
    full_name,
    team_name,
    total_laps,
    total_lap_duration,
    DENSE_RANK() OVER(PARTITION BY meeting_name ORDER BY total_lap_duration ASC) AS meetings_ranking
FROM
    meeting_driver_ranking
ORDER BY
    meeting_key ASC
;




------------Ejercicio 7: Ranking de pilotos por meeting (DE ACUERDO A LOS RESULTADOS DE https://www.formula1.com/en/results/2026/races/1285/canada/race-result)------------
with meeting_driver_ranking AS (
    SELECT
        get_json_object(dri.raw_json, '$.full_name') AS full_name,
        get_json_object(dri.raw_json, '$.team_name') AS team_name,
        get_json_object(met.raw_json, '$.meeting_key') AS meeting_key,
        get_json_object(met.raw_json, '$.meeting_official_name') AS meeting_name,
        get_json_object(ses.raw_json, '$.session_key') AS session_key,
        get_json_object(ses.raw_json, '$.session_name') AS session_name,
        ROUND(
            SUM(TRY_CAST(get_json_object(lap.raw_json, '$.lap_duration') AS DOUBLE)), 
        2) AS total_lap_duration,
        COUNT(get_json_object(lap.raw_json, '$.lap_number')) AS total_laps
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON get_json_object(lap.raw_json, '$.session_key') = get_json_object(ses.raw_json, '$.session_key')
    JOIN 
        workspace.formula_1.bronze_meetings met ON get_json_object(ses.raw_json, '$.meeting_key') = get_json_object(met.raw_json, '$.meeting_key')
    JOIN
        workspace.formula_1.bronze_drivers dri 
        ON get_json_object(lap.raw_json, '$.driver_number') = get_json_object(dri.raw_json, '$.driver_number')
        AND get_json_object(met.raw_json, '$.meeting_key') = get_json_object(dri.raw_json, '$.meeting_key')
    WHERE
        get_json_object(ses.raw_json, '$.meeting_key') = '1285'
        -- AND
        -- get_json_object(ses.raw_json, '$.session_name') = 'Sprint Qualifying'
        AND
        get_json_object(lap.raw_json, '$.is_pit_out_lap') = 'false'
    GROUP BY
        full_name,
        team_name,
        meeting_key,
        meeting_name,
        session_key,
        session_name
    -- HAVING
    --     total_laps > 50
)
SELECT
    meeting_name,
    session_name,
    full_name,
    team_name,
    total_laps,
    total_lap_duration,
    DENSE_RANK() OVER(PARTITION BY session_name ORDER BY total_lap_duration ASC) AS meetings_ranking
FROM
    meeting_driver_ranking
ORDER BY
    meeting_key ASC
;

-- SELECT * FROM formula_1.silver_dim_meetings_sessions WHERE meeting_key = 1285;